# 03 — Code Reviewer Agent: 4-Node Sequential Pipeline

### Core concepts covered
| Concept | Description |
|---------|-------------|
| Sequential pipeline | 4 nodes chained one after another |
| One LLM call per node | Each node does one focused analysis task |
| State accumulation | State grows richer as it moves through nodes |

### Graph flow
```
START → analyze_bugs → suggest_improvements → score_quality → write_report → END
```

> **Before running:** set `GROQ_API_KEY` in a `.env` file in this folder.

## 1. Install Dependencies

In [ ]:
# !pip install langgraph langchain langchain-groq python-dotenv

## 2. Imports & Setup

In [ ]:
from typing import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model("groq:llama3-8b-8192")

## 3. Define the State

In [ ]:
class ReviewState(TypedDict):
    code:          str   # source code to review
    language:      str   # programming language
    bugs:          str   # output from Node 1
    suggestions:   str   # output from Node 2
    quality_score: int   # output from Node 3
    report:        str   # final compiled report

## 4. Define the Nodes

Each node does **one job only** and adds its result to the state.

In [ ]:
def analyze_bugs(state: ReviewState) -> dict:
    """Node 1 — scan the code for bugs and runtime errors."""
    print("  🔍 Scanning for bugs...")
    prompt = (
        f"Analyze this {state['language']} code strictly for bugs, logic errors, "
        f"and runtime issues:\n\n"
        f"```{state['language']}\n{state['code']}\n```\n\n"
        "List only ACTUAL bugs. If none, say \"No bugs found.\"\n"
        "Be concise — one line per issue."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"bugs": response.content.strip()}

In [ ]:
def suggest_improvements(state: ReviewState) -> dict:
    """Node 2 — suggest 3–5 concrete code improvements."""
    print("  💡 Generating improvement suggestions...")
    prompt = (
        f"Review this {state['language']} code and give 3–5 concrete improvement suggestions:\n"
        "- Readability / naming\n- Performance\n- Best practices / edge cases\n\n"
        f"```{state['language']}\n{state['code']}\n```\n\n"
        "Number each suggestion. Be specific and brief."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"suggestions": response.content.strip()}

In [ ]:
def score_quality(state: ReviewState) -> dict:
    """Node 3 — give a numeric quality score from 1 to 10."""
    print("  📊 Scoring code quality...")
    prompt = (
        f"Rate the overall quality of this {state['language']} code on a scale of 1 to 10.\n"
        "Consider: correctness, readability, efficiency, naming, and structure.\n\n"
        f"```{state['language']}\n{state['code']}\n```\n\n"
        "Reply with a SINGLE integer between 1 and 10. Nothing else."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    try:
        raw   = "".join(ch for ch in response.content if ch.isdigit())
        score = max(1, min(10, int(raw[:2])))
    except Exception:
        score = 5
    return {"quality_score": score}

In [ ]:
def write_report(state: ReviewState) -> dict:
    """Node 4 — compile everything into a formatted code review report."""
    print("  📝 Writing report...\n")
    score  = state["quality_score"]
    filled = "⭐" * score
    empty  = "☆" * (10 - score)

    report = (
        f"\n{'═'*52}\n"
        f"            CODE REVIEW REPORT\n"
        f"{'═'*52}\n"
        f"  Language      : {state['language']}\n"
        f"  Quality Score : {filled}{empty}  ({score}/10)\n"
        f"{'─'*52}\n"
        f"  BUGS FOUND\n{'─'*52}\n"
        f"{state['bugs']}\n\n"
        f"{'─'*52}\n"
        f"  IMPROVEMENT SUGGESTIONS\n{'─'*52}\n"
        f"{state['suggestions']}\n"
        f"{'═'*52}"
    )
    print(report)
    return {"report": report}

## 5. Build & Compile the Graph

In [ ]:
builder = StateGraph(ReviewState)

builder.add_node("analyze_bugs",         analyze_bugs)
builder.add_node("suggest_improvements", suggest_improvements)
builder.add_node("score_quality",        score_quality)
builder.add_node("write_report",         write_report)

builder.add_edge(START,                  "analyze_bugs")
builder.add_edge("analyze_bugs",         "suggest_improvements")
builder.add_edge("suggest_improvements", "score_quality")
builder.add_edge("score_quality",        "write_report")
builder.add_edge("write_report",         END)

graph = builder.compile()
print("Graph compiled!")

## 6. Visualise the Graph *(optional)*

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualisation skipped: {e}")

## 7. Review Code

Edit `CODE_TO_REVIEW` and `LANGUAGE` below to review your own code.

In [ ]:
# ── Edit this to review your own code ──────────────────────────
CODE_TO_REVIEW = """
def fetch_user_data(user_id):
    import requests
    url = "https://api.example.com/users/" + user_id
    response = requests.get(url)
    data = response.json()
    name = data['name']
    age  = data['age']
    if age > 18:
        print("Adult user: " + name)
    else:
        print("Minor user: " + name)
    return data

fetch_user_data(123)
"""
LANGUAGE = "Python"
# ────────────────────────────────────────────────────────────────

print(f"Reviewing the following {LANGUAGE} code:\n")
print("─" * 40)
print(CODE_TO_REVIEW.strip())
print("─" * 40)

In [ ]:
graph.invoke({
    "code":          CODE_TO_REVIEW,
    "language":      LANGUAGE,
    "bugs":          "",
    "suggestions":   "",
    "quality_score": 0,
    "report":        "",
})